# Global Health Progress, 2000–2019: A Data Story from the World Bank's Health, Nutrition and Population Statistics

**3520872** | CSCU9M3 — Scripting for Data Science | Autumn 2026

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Colab always has a module called google.colab loaded automatically, so I
# can use that to check if the notebook is running in Colab or not, and
# point at the right folder for the data either way.
running_in_colab = "google.colab" in sys.modules

if running_in_colab:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Untitled folder/data/hnp_stats"
else:
    DATA_DIR = "data/hnp_stats"

## Executive Summary

_[Written last, after the findings below are known — max 200 words.]_

## Introduction

_[Written last, after the findings below are known — max 200 words.]_

## Data Cleaning

The World Bank's Health, Nutrition and Population Statistics (HNP) data comes as several linked CSV files: a main data file (one row per country/indicator, one column per year 1960–2021), a country metadata file, and an indicator metadata file. The main file needs cleaning before any analysis. It mixes real countries with regional and income-group aggregates in the same rows, it contains hundreds of indicators and six decades of years that this report does not use, and many country/indicator combinations have no data at all for the years we care about.

In [ ]:
data = pd.read_csv(f"{DATA_DIR}/HNP_StatsData.csv")
country_info = pd.read_csv(f"{DATA_DIR}/HNP_StatsCountry.csv")

# Both files end every line with a trailing comma, so pandas reads an extra
# empty column called "Unnamed: xx" at the end. I get rid of those columns
# before doing anything else with the data.
extra_columns = [col for col in data.columns if col.startswith("Unnamed")]
data = data.drop(columns=extra_columns)

extra_columns = [col for col in country_info.columns if col.startswith("Unnamed")]
country_info = country_info.drop(columns=extra_columns)

print("Main data file:", data.shape)
print("Country info file:", country_info.shape)

Main data file: (117838, 66)
Country info file: (265, 30)


Removing non-country aggregates. Rows such as "Africa Eastern and Southern" or "High income" are regional/income-group aggregates, not countries, and would distort any country-level comparison (e.g. a regional aggregate being counted alongside the countries that make it up). An alternative would be to hand-pick a list of aggregate names to drop, but this was rejected because it is brittle and error-prone, it relies on recognising every aggregate by hand and would silently miss any that are mis-spelled or renamed. The country metadata file already distinguishes real countries from aggregates, every real country has a Region value, while every aggregate has a blank one. Filtering on that column is a rule based on the data rather than a manual list.

In [ ]:
# Rows with no Region are not real countries, they are totals for a whole
# region or income group (e.g. "Africa Eastern and Southern"), so I filter
# those out here and keep just the columns I need from the country info.
real_countries = country_info[country_info["Region"].notna()]
real_countries = real_countries[["Country Code", "Region", "Income Group"]]

data = data.merge(real_countries, on="Country Code", how="inner")

print("Rows after removing region/income totals:", len(data))
print("Number of countries left:", data["Country Code"].nunique())

Rows after removing region/income totals: 96131
Number of countries left: 217


Choosing indicators. The full dataset covers hundreds of indicators spanning population, nutrition, disease, and health systems, far more than a 1500-word report can cover. Five indicators were chosen to address one question (has global health improved since 2000, and is that improvement shared equally?), life expectancy at birth, under-5 mortality, measles immunisation coverage, fertility rate, and current health expenditure (% of GDP). These were chosen for good data coverage (checked below), and because together they cover outcomes (mortality, life expectancy), a driver (health spending), and context (fertility, immunisation).

In [ ]:
indicators_wanted = {
    "SP.DYN.LE00.IN": "Life expectancy at birth (years)",
    "SH.DYN.MORT": "Under-5 mortality (per 1,000 live births)",
    "SH.IMM.MEAS": "Measles immunisation (% of children 12-23 months)",
    "SP.DYN.TFRT.IN": "Fertility rate (births per woman)",
    "SH.XPD.CHEX.GD.ZS": "Current health expenditure (% of GDP)",
}

data = data[data["Indicator Code"].isin(indicators_wanted)]
data["Indicator Name"] = data["Indicator Code"].map(indicators_wanted)
print("Rows after keeping only the 5 indicators I want:", len(data))

Rows after keeping only the 5 indicators I want: 1085


Choosing the year range. Current health expenditure (% of GDP) is only reported from 2000 onwards (an earlier, broader methodology didn't track it), so 2000 is the start of the comparison across all five indicators, using an earlier start would mean comparing on four indicators for most of the window and five only at the end. 2019 is used as the end point rather than the most recent years (2020-2021) because COVID-19 caused sharp, temporary disruption to both health outcomes and data collection in those years, including them would make a "has health improved" comparison partly about the pandemic rather than the longer-run trend this report is about.

In [ ]:
YEARS = [str(year) for year in range(2000, 2020)]
columns_to_keep = ["Country Name", "Country Code", "Region", "Income Group",
                    "Indicator Name", "Indicator Code"] + YEARS
data = data[columns_to_keep]
print("Columns kept:", list(data.columns))

Columns kept: ['Country Name', 'Country Code', 'Region', 'Income Group', 'Indicator Name', 'Indicator Code', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']


Handling missing values. For the before/after comparisons used later in this report, a country/indicator row is only usable if it has a value in both endpoint years, 2000 and 2019. Rows missing either endpoint are dropped rather than filled in by interpolation, interpolating would invent a trend for a country that never reported that indicator reliably, and that gap in reporting is a real feature of the data rather than something to paper over.

In [ ]:
missing_2000 = data["2000"].isna().sum()
missing_2019 = data["2019"].isna().sum()
print("Rows missing a 2000 value:", missing_2000)
print("Rows missing a 2019 value:", missing_2019)

clean_data = data.dropna(subset=["2000", "2019"])
clean_data = clean_data.reset_index(drop=True)

percent_kept = round(len(clean_data) / len(data) * 100)
print("Rows kept:", len(clean_data), "out of", len(data), f"({percent_kept}%)")

Rows missing a 2000 value: 117
Rows missing a 2019 value: 114
Rows kept: 961 out of 1085 (89%)


In [ ]:
clean_data[["Country Name", "Region", "Income Group", "Indicator Name", "2000", "2019"]].head()

  Country Name                 Region         Income Group  \
0  Afghanistan             South Asia           Low income   
1  Afghanistan             South Asia           Low income   
2  Afghanistan             South Asia           Low income   
3  Afghanistan             South Asia           Low income   
4      Albania  Europe & Central Asia  Upper middle income   

                                      Indicator Name     2000    2019  
0                  Fertility rate (births per woman)    7.485   4.321  
1  Measles immunisation (% of children 12-23 months)   27.000  64.000  
2                   Life expectancy at birth (years)   55.841  64.833  
3          Under-5 mortality (per 1,000 live births)  129.200  60.100  
4                  Fertility rate (births per woman)    2.157   1.597  

clean_data is the dataset used for the rest of this report: real countries only, five indicators, years 2000–2019, with both endpoint years guaranteed present for every row. Intermediate years (2001–2018) may still contain gaps for some countries, this is fine because only the Exploratory Data Analysis section uses them, as year-by-year averages rather than individual country values.

## Exploratory Data Analysis

_[Phase 2 — to be completed next.]_

## Descriptive Analytics

_[Phase 3 — to be completed next.]_

## Suggestions

_[Phase 4 — to be completed next.]_

## Bibliography

_[Phase 6 — to be completed last.]_